# Converting cubed sphere to lat/lon grid using JEDI

## Prerequisite 

Please review [here](https://mer-a-o.github.io/howtojedi/discover/) to learn about setting up your work environment and configuring the `jedi-bundle`. 

Before starting, verify:
- [ ] `$JEDI_BUILD` environment variable is set
- [ ] `fv3jedi_converttolatlon.x` exists in `$JEDI_BUILD/fv3jedi_converttolatlon.x`
- [ ] Input files exist in the `jedi_utils/run_convert_to_latlon/inputs` directory (background, observation, and auxiliary files such as geometry configurations)
- [ ] You have access to compute resources (interactive node or batch system)

---

## Usage and limitations
`fv3jedi_converttolatlon.x` can be used to convert a cubed sphere file to a lat/lon grid. However, the user needs to specify which model level they want to perform the conversion on. This means that the executable doesn't automatically regrid all the model levels.

An example of an input YAML file is below.
You will need to specify:
- Input geometry (under `state geometry`)
- Details about the input file (under `state`)
- Details about the `latlon interpolation`. More info here: ???? 

### YAML structure:

```yaml
input geometry:
  fms initialization:
    namelist filename: Data/fv3files/fmsmpp.nml
    field table filename: Data/fv3files/field_table_gmao
  akbk: Data/inputs/fv3files/akbk72.nc4
  npx: 13
  npy: 13
  npz: 72
output geometry:
  akbk: Data/inputs/fv3files/akbk72.nc4
  npx: 25
  npy: 25
  npz: 72
states:
- input:
    datetime: 2020-09-03T18:00:00Z
    filetype: cube sphere history
    provider: geos
    datapath: Data/inputs/geos_c12
    filename: geos_cf.bkg.%yyyy%mm%dd_%hh%MM%ssz.nc4
    state variables:
    - air_pressure_thickness
    - volume_mixing_ratio_of_no2
    - volume_mixing_ratio_of_no
    - volume_mixing_ratio_of_o3
    - air_pressure_at_surface
    - water_vapor_mixing_ratio_wrt_moist_air
    field io names: &field_io_names
      air_pressure_thickness: DELP
      volume_mixing_ratio_of_no2: NO2
      volume_mixing_ratio_of_no: "NO"
      volume_mixing_ratio_of_o3: O3
      air_pressure_at_surface: PS
      water_vapor_mixing_ratio_wrt_moist_air: SPHU
  output:
   filetype: cube sphere history
   provider: geos
   datapath: Data/
   filename: geos_cf.bkg.converted.%yyyy%mm%dd_%hh%MM%ssz.nc4
   field io names: *field_io_names
test:
  reference filename: testoutput/convertstate_geos_cf.ref
  test output filename: testoutput/convertstate_geos_cf.test.out
```

```yaml
states to latlon:
- state geometry:
    fms initialization:
      namelist filename: geometry_input/fmsmpp.nml
      field table filename: geometry_input/field_table_gmao
    akbk: geometry_input/akbk72.nc4
    npx: 13
    npy: 13
    npz: 72
    field metadata override: geometry_input/geos_cf.yaml

  state:
    datetime: 2020-09-03T15:00:00Z
    filetype: cube sphere history
    state variables: [DELP,NO2,NO,O3,PS,SPHU]
    datapath: geos_c12/
    filename: geos_cf.bkg.20200903_150000z.nc4

  latlon interpolation:
    local interpolator type: oops unstructured grid interpolator
    resolution in degrees: 5.0  # low resolution for testing
    # need to use long names (see "field metadata override" file)
    variables to output: [air_pressure_thickness,volume_mixing_ratio_of_no2]
    pressure levels in hPa: [925, 850, 700, 500, 250]
    model levels: [71, 10]
    bottom model level: true
    datapath: output
    prefix: latlon.geos
    
```

### Running convertstate

```bash
$MPIEXEC "-n" "6" $JEDI_BUILD/bin/fv3jedi_converttolatlon.x converttolatlon_geos.yaml 2>&1 | tee log_convert_to_latlon.txt
```

In this example, two files will be generated. The "PressureLevels" file will include the `variables to output` values in latlon grid for levels specified under `pressure levels in hPa`. Similarly, the "modelLevels" file will include the `variables to output` values in latlon grid for levels specified under `model levels`.

---

Another example of this application is ctest `fv3jedi_test_tier1_converttolatlon_gfs` in [fv3-jedi](https://github.com/JCSDA/fv3-jedi/blob/1.8.0/test/CMakeLists.txt#L992). In this [example](https://github.com/JCSDA/fv3-jedi/blob/1.8.0/test/testinput/converttolatlon_gfs.yaml), filetype of `fms restart` is converted from cubed sphere to latlon grid. 